# Week 8: 整数规划

## 学习目标

1. 理解整数规划的特点和应用场景
2. 掌握 0-1 规划和混合整数规划
3. 学会处理逻辑约束
4. 理解整数规划的求解复杂性

## 1. 整数规划基础

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pulp import *

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("整数规划工具已加载")

### 1.1 整数规划 vs 线性规划

**关键区别：**
- 线性规划：变量可以是任意实数
- 整数规划：变量必须是整数
- 混合整数规划：部分变量是整数
- 0-1 规划：变量只能取 0 或 1

In [ ]:
# 示例：对比线性规划松弛和整数规划
# 最大化 z = 3x + 4y
# 约束：
#   x + 2y ≤ 7
#   3x + y ≤ 8
#   x, y ≥ 0 且为整数

# 线性规划松弛
prob_lp = LpProblem("LP_Relaxation", LpMaximize)
x_lp = LpVariable("x", lowBound=0)
y_lp = LpVariable("y", lowBound=0)

prob_lp += 3*x_lp + 4*y_lp
prob_lp += x_lp + 2*y_lp <= 7
prob_lp += 3*x_lp + y_lp <= 8
prob_lp.solve()

# 整数规划
prob_ip = LpProblem("Integer_Program", LpMaximize)
x_ip = LpVariable("x", lowBound=0, cat='Integer')
y_ip = LpVariable("y", lowBound=0, cat='Integer')

prob_ip += 3*x_ip + 4*y_ip
prob_ip += x_ip + 2*y_ip <= 7
prob_ip += 3*x_ip + y_ip <= 8
prob_ip.solve()

print("线性规划松弛 vs 整数规划")
print("=" * 40)
print("线性规划松弛:")
print(f"  x = {x_lp.varValue:.2f}, y = {y_lp.varValue:.2f}")
print(f"  目标值 = {value(prob_lp.objective):.2f}")
print("\n整数规划:")
print(f"  x = {x_ip.varValue}, y = {y_ip.varValue}")
print(f"  目标值 = {value(prob_ip.objective):.2f}")
print(f"\n差距 = {value(prob_lp.objective) - value(prob_ip.objective):.2f}")

In [ ]:
# 可视化可行域
fig, ax = plt.subplots(figsize=(10, 8))

# 可行域边界
x = np.linspace(0, 3, 100)
y1 = (7 - x) / 2
y2 = 8 - 3*x

ax.plot(x, y1, 'r-', label='x + 2y = 7')
ax.plot(x, y2, 'b-', label='3x + y = 8')

# 整数点
integer_points = []
for i in range(4):
    for j in range(4):
        if i + 2*j <= 7 and 3*i + j <= 8:
            integer_points.append((i, j))
            z = 3*i + 4*j
            ax.plot(i, j, 'go', markersize=10)

# LP 松弛最优解
ax.plot(x_lp.varValue, y_lp.varValue, 'r^', markersize=12, label=f'LP 最优 ({x_lp.varValue:.2f}, {y_lp.varValue:.2f})')

# 整数最优解
ax.plot(x_ip.varValue, y_ip.varValue, 'bs', markersize=12, label=f'IP 最优 ({x_ip.varValue:.0f}, {y_ip.varValue:.0f})')

ax.set_xlim(0, 3.5)
ax.set_ylim(0, 4)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('整数规划可行域')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 0-1 规划

### 2.1 背包问题

In [ ]:
# 0-1 背包问题
# 目标：选择物品最大化价值，不超过容量限制

items = ['A', 'B', 'C', 'D', 'E']
values = {'A': 10, 'B': 8, 'C': 15, 'D': 12, 'E': 6}
weights = {'A': 3, 'B': 2, 'C': 5, 'D': 4, 'E': 2}
capacity = 10

# 建模
prob = LpProblem("Knapsack", LpMaximize)

# 0-1 变量：是否选择该物品
x = LpVariable.dicts("select", items, cat='Binary')

# 目标函数
prob += lpSum([values[i] * x[i] for i in items]), "Total_Value"

# 容量约束
prob += lpSum([weights[i] * x[i] for i in items]) <= capacity, "Capacity"

# 求解
prob.solve()

print("背包问题求解结果")
print("=" * 40)
print("选择的物品:")
total_weight = 0
total_value = 0
for i in items:
    if x[i].varValue == 1:
        print(f"  {i}: 价值 {values[i]}, 重量 {weights[i]}")
        total_weight += weights[i]
        total_value += values[i]

print(f"\n总价值: {total_value}")
print(f"总重量: {total_weight} / {capacity}")

### 2.2 设施选址问题

In [ ]:
# 设施选址问题
# 决定在哪些地点建立仓库，满足所有客户需求

# 候选位置
locations = ['L1', 'L2', 'L3']
clients = ['C1', 'C2', 'C3', 'C4']

# 固定成本
fixed_cost = {'L1': 100, 'L2': 120, 'L3': 80}

# 运输成本（位置 -> 客户）
transport_cost = {
    ('L1', 'C1'): 4, ('L1', 'C2'): 6, ('L1', 'C3'): 8, ('L1', 'C4'): 5,
    ('L2', 'C1'): 5, ('L2', 'C2'): 3, ('L2', 'C3'): 4, ('L2', 'C4'): 7,
    ('L3', 'C1'): 7, ('L3', 'C2'): 5, ('L3', 'C3'): 3, ('L3', 'C4'): 4
}

# 客户需求
demand = {'C1': 50, 'C2': 40, 'C3': 60, 'C4': 30}

# 建模
prob = LpProblem("Facility_Location", LpMinimize)

# 变量
y = LpVariable.dicts("open", locations, cat='Binary')  # 是否开放
x = LpVariable.dicts("ship", [(l, c) for l in locations for c in clients], lowBound=0)

# 目标：最小化固定成本 + 运输成本
prob += lpSum([fixed_cost[l] * y[l] for l in locations]) + \
        lpSum([transport_cost[(l, c)] * x[(l, c)] for l in locations for c in clients])

# 约束：满足客户需求
for c in clients:
    prob += lpSum([x[(l, c)] for l in locations]) == demand[c]

# 约束：只能从开放的仓库发货
for l in locations:
    for c in clients:
        prob += x[(l, c)] <= demand[c] * y[l]

# 求解
prob.solve()

print("设施选址问题求解结果")
print("=" * 40)
print("开放仓库:")
for l in locations:
    if y[l].varValue == 1:
        print(f"  {l}: 固定成本 {fixed_cost[l]}")

print(f"\n总成本: {value(prob.objective):.0f}")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 开放决策
open_status = [y[l].varValue for l in locations]
axes[0].bar(locations, open_status, color=['green' if s == 1 else 'red' for s in open_status])
axes[0].set_ylabel('开放状态')
axes[0].set_title('仓库开放决策')
axes[0].set_ylim(0, 1.2)

# 运输量
ship_matrix = np.zeros((len(locations), len(clients)))
for i, l in enumerate(locations):
    for j, c in enumerate(clients):
        ship_matrix[i, j] = x[(l, c)].varValue

import seaborn as sns
sns.heatmap(ship_matrix, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=clients, yticklabels=locations, ax=axes[1])
axes[1].set_xlabel('客户')
axes[1].set_ylabel('仓库')
axes[1].set_title('运输量分配')

plt.tight_layout()
plt.show()

## 3. 逻辑约束

In [ ]:
# 逻辑约束示例
# 问题：项目选择，存在相互依赖或排斥

projects = ['P1', 'P2', 'P3', 'P4', 'P5']
profits = {'P1': 20, 'P2': 15, 'P3': 25, 'P4': 10, 'P5': 30}
costs = {'P1': 8, 'P2': 6, 'P3': 12, 'P4': 5, 'P5': 15}
budget = 30

prob = LpProblem("Project_Selection", LpMaximize)

x = LpVariable.dicts("select", projects, cat='Binary')

# 目标
prob += lpSum([profits[p] * x[p] for p in projects])

# 预算约束
prob += lpSum([costs[p] * x[p] for p in projects]) <= budget

# 逻辑约束
# 1. 如果选择 P3，必须选择 P1
prob += x['P3'] <= x['P1'], "P3_implies_P1"

# 2. P2 和 P4 不能同时选择
prob += x['P2'] + x['P4'] <= 1, "P2_or_P4"

# 3. 至少选择 P2 或 P5
prob += x['P2'] + x['P5'] >= 1, "P2_or_P5_required"

# 求解
prob.solve()

print("项目选择问题（含逻辑约束）")
print("=" * 40)
print("选择的项目:")
total_cost = 0
total_profit = 0
for p in projects:
    if x[p].varValue == 1:
        print(f"  {p}: 利润 {profits[p]}, 成本 {costs[p]}")
        total_cost += costs[p]
        total_profit += profits[p]

print(f"\n总利润: {total_profit}")
print(f"总成本: {total_cost} / {budget}")

## 4. 单车站点选址

In [ ]:
# 问题：选择 k 个站点建立共享单车点
# 目标：最大化覆盖人口

# 候选位置
candidates = list(range(1, 11))  # 10 个候选位置

# 各位置可覆盖的人口（百人）
coverage = {1: 15, 2: 12, 3: 20, 4: 8, 5: 18, 
            6: 10, 7: 22, 8: 14, 9: 16, 10: 11}

# 站点数量限制
k = 4

# 建模
prob = LpProblem("Station_Placement", LpMaximize)

x = LpVariable.dicts("place", candidates, cat='Binary')

# 目标
prob += lpSum([coverage[i] * x[i] for i in candidates])

# 站点数量约束
prob += lpSum([x[i] for i in candidates]) == k

# 求解
prob.solve()

print("单车站点选址结果")
print("=" * 40)
print(f"选择建立 {k} 个站点:")

selected = []
for i in candidates:
    if x[i].varValue == 1:
        selected.append(i)
        print(f"  位置 {i}: 覆盖人口 {coverage[i]}00 人")

total_coverage = sum(coverage[i] for i in selected)
print(f"\n总覆盖人口: {total_coverage}00 人")

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(len(candidates))
colors = ['green' if x[i].varValue == 1 else 'lightgray' for i in candidates]

bars = ax.bar(x_pos, [coverage[i] for i in candidates], color=colors, edgecolor='black')

ax.set_xlabel('候选位置')
ax.set_ylabel('覆盖人口（百人）')
ax.set_title(f'单车站点选址（选择 {k} 个）')
ax.set_xticks(x_pos)
ax.set_xticklabels(candidates)

# 标记选择的站点
for i, bar in enumerate(bars):
    if x[candidates[i]].varValue == 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                '✓', ha='center', fontsize=14, color='green')

ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Research Thinking

### 问题 1：整数规划的复杂性

为什么整数规划比线性规划难？

**回答：**

1. **NP-hard 问题**：没有多项式时间算法
2. **搜索空间大**：2^n 种组合
3. **松弛间隙**：LP 松弛解可能离整数解很远
4. **实际影响**：大规模问题求解时间可能很长

### 问题 2：分支定界法

整数规划求解器如何工作？

**回答：**

1. **LP 松弛**：先求解连续变量版本
2. **分支**：对非整数解创建子问题
3. **定界**：用上下界剪枝
4. **迭代**：直到找到最优整数解

### 问题 3：启发式方法

大规模整数规划怎么办？

**回答：**

- **启发式算法**：贪心、局部搜索
- **元启发式**：遗传算法、模拟退火
- **近似算法**：保证一定质量的解
- **分解方法**：将大问题分解为小问题

## 6. 练习

### 练习 1
扩展背包问题，增加体积约束。

In [ ]:
# 你的代码


### 练习 2
设计一个排班问题，每个员工最多工作一定天数。

In [ ]:
# 你的代码


### 练习 3
实现一个集合覆盖问题。

In [ ]:
# 你的代码


## 7. 总结

### 本周学习要点

1. **整数规划特点**：离散决策变量
2. **0-1 规划**：二选一决策
3. **逻辑约束**：用数学表达逻辑关系
4. **求解复杂性**：NP-hard 问题

### 关键洞察

- 整数规划能表达更复杂的决策
- 逻辑约束是建模的强大工具
- 大规模问题需要启发式方法
- 权衡求解时间和解的质量